# Notebook 06 — Active Learning & Surveillance Priority Queue

## Goal
Solve the rural groundwater testing resource-allocation bottleneck:
- Under fixed physical laboratory budgets (e.g., 50 tests/month), **which wells should be tested next?**
- Implement **Shannon Entropy** epistemic uncertainty sampling
- Formulate a **Multi-Objective Hybrid Acquisition Function** (Uncertainty + BBN Risk + Anomaly Urgency)
- Simulate a multi-round field campaign comparing **Active Learning vs. Uniform Random Sampling**
- Generate the prioritized field surveillance testing queue (`testing_priority_queue.csv`)

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
from IPython.display import Image, display

from src.active_learning import (
    load_pipeline_data,
    calculate_entropy,
    compute_acquisition_scores,
    simulate_active_learning,
    run_active_learning_pipeline
)

PROCESSED_DIR = '../data/processed'
FIGURES_DIR   = '../figures'

## Step 1 — Load Bayesian Enriched Wells & Compute Acquisition Scores
Evaluate Shannon Entropy $H(p) = - [p \log_2(p) + (1-p) \log_2(1-p)]$ and Hybrid Priority Scores.

In [ ]:
df = load_pipeline_data(PROCESSED_DIR)
scored_df = compute_acquisition_scores(df)

print("Priority Tier Distribution:")
print(scored_df['priority_tier'].value_counts())
scored_df[['station', 'state', 'district', 'pred_prob', 'uncertainty_entropy', 'testing_priority_score', 'priority_tier', 'testing_reason']].head(10)

## Step 2 — Sequential Field Surveillance Simulation (Active Learning vs Random)
Simulate 8 sequential monthly rounds (testing 50 wells per round, 400 tests total).
Compare how many contaminated wells each strategy discovers and how fast model generalization improves.

In [ ]:
feature_cols = ['latitude', 'longitude', 'groundwater_level_m', 'state_encoded', 'district_encoded']
sim_df = simulate_active_learning(
    df,
    feature_cols=feature_cols,
    batch_size=50,
    n_rounds=8,
    random_seed=42
)
sim_df[['round', 'tests_conducted', 'al_cum_contam', 'rnd_cum_contam', 'contam_discovery_delta', 'al_auc', 'rnd_auc']]

## Step 3 — Inspect Active Learning Performance Curves

In [ ]:
display(Image(filename=os.path.join(FIGURES_DIR, 'active_learning_curves.png')))

## Step 4 — Top Wells in Surveillance Priority Queue
Inspect the top candidates recommended for immediate laboratory sample collection.

In [ ]:
queue_df, sim_df, metrics = run_active_learning_pipeline(PROCESSED_DIR, figures_dir=FIGURES_DIR)

print(f"AL Contaminant Lift: +{metrics['contaminant_lift_pct']}% more contaminated wells detected!")
queue_cols = ['priority_rank', 'station', 'state', 'district', 'testing_priority_score', 'priority_tier', 'testing_reason']
queue_df[queue_cols].head(15)